<div style="text-align: center; padding: 30px; background-color: #f5f7fa; border-radius: 12px; box-shadow: 0px 4px 10px rgba(0,0,0,0.1);">

  <h1 style="color: #2c3e50; font-family: 'Helvetica Neue', sans-serif; font-size: 42px; margin-bottom: 10px;">
    GluN1/NR1 Multiple Sequence Alignment
  </h1>

  <h3 style="color: #7f8c8d; font-family: 'Georgia', serif; font-weight: normal; font-size: 22px; margin-top: 0;">
    Multiple Sequence Alignment and Comparison
  </h3>

  <p style="color: #34495e; font-family: 'Arial', sans-serif; font-size: 16px; margin-top: 20px;">
    Author: <b>Vallari Eastman</b> <br>
    Revision 1: <b>Sunday Nov 2nd, 2025</b>
  </p>
</div>

<div style="text-align: center; padding: 25px; background-color: #f5f7fa; border-radius: 12px; box-shadow: 0px 4px 8px rgba(0,0,0,0.08); margin-top: 30px;">

  <h2 style="color: #2c3e50; font-family: 'Helvetica Neue', sans-serif; font-size: 32px; margin-bottom: 10px;">
    BioPython Tutorial & Cookbook Documentation
  </h2>

  <h4 style="color: #7f8c8d; font-family: 'Georgia', serif; font-weight: normal; font-size: 20px; margin-top: 0;">
    GluN1/NR1 (UniProt: Q05586, PDB ID: 7EU8)<br>
  </h4>

  <p style="color: #34495e; font-family: 'Arial', sans-serif; font-size: 15px; margin-top: 18px;">
    <b>Key Questions Driving This Analysis:</b><br><br>
        1. What’s the taxonomic coverage and redundancy?<br>
        2. Which positions are highly conserved vs hypervariable? <br>
        <br>
  </p>

<div style="background-color:#f8f9fb; padding:25px; border-radius:12px; box-shadow:0px 4px 8px rgba(0,0,0,0.08);">

  <h2 style="color:#2c3e50; font-family:'Helvetica Neue',sans-serif; font-size:28px; margin-bottom:20px;">
    Methods:
  </h2>
  <h3 style="color:#7f8c8d; font-family:'Georgia',serif; font-weight:normal; font-size:22px; margin-top:0;">
    Sequence Used</b>: UniProt ID: Q05586 & Pfam Entry: PF10613
  </h3> 
   <p style="color: #34495e; font-family: 'Arial', sans-serif; font-size: 15px; margin-top: 18px;">
    The human GRIN1 gene product (UniProt ID: Q05586) was used as the query sequence to identify homologous proteins in the InterPro database. The InterPro web portal (https://www.ebi.ac.uk/interpro/
) was searched for Q05586, and the Pfam entry PF10613 (Ligated ion channel L-glutamate- and glycine-binding site) was selected. Under the Alignment tab, the Full Alignment file was downloaded in compressed format (PF10613.alignment.full.gz). The file was decompressed using the Unix command line (gunzip PF10613.alignment.full.gz) and converted to FASTA format using Biopython.
       <br><br>
The aligned FASTA file was filtered to retain the top 100 sequences most similar to the human target sequence (Q05586), including the target itself. Pairwise sequence identity (PID) and coverage relative to the target were computed, and sequences with ≥0.9 coverage were retained. The selected dataset was then cleaned and annotated using UniProt species codes and common names derived from speclist.txt, ensuring standardized species labels and removal of incomplete or highly gapped sequences.
       <br><br>
The final cleaned alignment (PF10613_top100.clean.aln) was converted to Phylip-relaxed format for downstream evolutionary analysis. Sequence redundancy was assessed by computing Henikoff weights and calculating the effective number of sequences per alignment column (Neff/L) to evaluate information content and diversity. A neighbor-joining phylogenetic tree was constructed using pairwise distances derived from the BLOSUM62 substitution model. 
       <br><br>
Sequence conservation was quantified using Shannon entropy at each alignment position. Columns were classified as conserved, moderate, or variable based on normalized entropy thresholds. Rolling-mean entropy (window size = 5) was used to identify the most and least conserved contiguous regions. A multiple sequence alignment visualization was generated to illustrate conserved motifs and hypervariable regions. Finally, a custom substitution matrix was generated from the alignment by calculating observed substitution frequencies for polar charged residues (D, E, H, K, R), normalized against expected independent frequencies, and converted into log-odds scores (bits). This matrix was validated by aligning short peptide pairs using Biopython’s PairwiseAligner to demonstrate consistency with expected biochemical substitution patterns.</p>
</div>

# Step 1: Fetch PFAM Stockholm File

1. Search Q05586 on InterPro Database
2. Toggle tab to Pfam
3. Choose: PF10613 Ligated ion channel L-glutamate- and glycine-binding site
4. Toggle Alignment Tab & download the FULL File
5. Run this in command line: gunzip PF10613.alignment.full.gz

In [130]:
# Makes sure target (Q05586) is in file
from Bio import SeqIO

def record_has_accession(rec, acc="Q05586"):
    acc = acc.upper()
    # 1) Obvious string fields
    haystacks = [rec.id or "", rec.name or "", rec.description or ""]
    if any(acc in h.upper() for h in haystacks):
        return True

    # 2) Parsed annotations from Stockholm (#=GS lines)
    ann = rec.annotations or {}
    # common places: 'accession' (string), 'accessions' (list), 'AC' (string), etc.
    for key in ("accession", "AC"):
        v = ann.get(key)
        if isinstance(v, str) and acc in v.upper():
            return True
    if isinstance(ann.get("accessions"), (list, tuple)):
        if any(acc in str(x).upper() for x in ann["accessions"]):
            return True

    # 3) dbxrefs sometimes carries accessions
    if any(acc in str(x).upper() for x in (rec.dbxrefs or [])):
        return True

    # 4) Pfam-style IDs: ACC_SPECIES/START-END
    core = rec.id.split("_", 1)[0]  # 'Q05586_HUMAN/...' -> 'Q05586'
    if core.upper().startswith(acc):
        return True

    return False

path = "PF10613.alignment.full"  # Stockholm file
# path = "PF10613_top100.aligned.fasta"
found = False
for rec in SeqIO.parse(path, "stockholm"):
    if record_has_accession(rec, "Q05586"):
        print(f" Found Q05586 in record: {rec.id}")
        found = True
        break

if not found:
    print(" Q05586 not found via parser fields — doing a raw text scan next...")
    with open(path, "r", errors="ignore") as fh:
        if "Q05586" in fh.read():
            print("Q05586 text is present in the file (parser didn’t expose it on this record).")
        else:
            print(" Q05586 not present in file text.")


 Found Q05586 in record: NMDZ1_HUMAN/444-546


# Step 2: Convert to FASTA

In [132]:
# Convert to Fasta
from Bio import AlignIO
AlignIO.convert("PF10613.alignment.full", "stockholm", "PF10613.full.fasta", "fasta")

1

# Step 3: Select Relevant Data (Get similarity scores and take the top 100)

In [103]:
# Uses QQ05586 as target, computes %ID vs target, then writes top N (including target)
from Bio import AlignIO, SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
from Bio.Align import MultipleSeqAlignment

GAPS = {"-", "."}

def _ungap(s: str) -> str:
    return s.replace("-", "").replace(".", "")

def _find_target_index(aln, target_hint=None):
    prefs = []
    if target_hint:
        prefs.append(str(target_hint).upper())
    prefs += ["NMDZ1_HUMAN", "Q05586"]
    for pref in prefs:
        for i, rec in enumerate(aln):
            text = " ".join([rec.id, getattr(rec, "name", ""), getattr(rec, "description", "")]).upper()
            if pref in text:
                return i
    return 0

def _pid_cov_vs_target(target_seq, other_seq):
    t = str(target_seq); o = str(other_seq)
    assert len(t) == len(o), "Alignment sequences must be same length."
    idx = [i for i, aa in enumerate(t) if aa not in GAPS]
    if not idx:
        return 0.0, 0.0, 1.0
    covered = matches = 0
    for i in idx:
        if o[i] not in GAPS:
            covered += 1
            if t[i].upper() == o[i].upper():
                matches += 1
    coverage = covered / len(idx)
    pid = (matches / covered) if covered else 0.0
    gap_frac = sum(1 for c in o if c in GAPS) / len(o) if o else 1.0
    return pid, coverage, gap_frac

def topN_from_msa(msa_file, target_hint=None, out_fasta=None, out_aligned=None,
                  N=100, min_coverage=0.90, include_target=True):
    """
    Select top N sequences by %ID to the target (over target non-gap columns).
    Guaranteed to include the target when include_target=True (immune to dedup/coverage filters).
    """
    aln = AlignIO.read(msa_file, "fasta")

    # --- find target ---
    t_idx = _find_target_index(aln, target_hint=target_hint)
    target_rec = aln[t_idx]
    t_pid, t_cov, t_gap = _pid_cov_vs_target(target_rec.seq, target_rec.seq)  # (1.0, 1.0, gap_frac)
    target_info = {
        "i": t_idx,
        "rec": target_rec,
        "pid": t_pid,
        "cov": t_cov,
        "gap_lightness": 1.0 - t_gap,
        "is_target": True,
    }

    # --- score all (except we won't filter out target later) ---
    scored = []
    for i, rec in enumerate(aln):
        pid, cov, gap_frac = _pid_cov_vs_target(target_rec.seq, rec.seq)
        if i == t_idx:
            continue  # handle target explicitly
        if cov >= min_coverage:
            scored.append({
                "i": i,
                "rec": rec,
                "pid": pid,
                "cov": cov,
                "gap_lightness": 1.0 - gap_frac,
                "is_target": False,
            })

    # --- sort ---
    scored.sort(key=lambda s: (s["pid"], s["cov"], s["gap_lightness"], -s["i"]), reverse=True)

    # --- dedup with target forced in first ---
    selected = []
    seen = set()

    if include_target:
        t_ung = _ungap(str(target_rec.seq)).upper()
        if t_ung:  # add target if not empty
            selected.append(target_info)
            seen.add(t_ung)

    # fill the rest up to N
    for s in scored:
        ung = _ungap(str(s["rec"].seq)).upper()
        if not ung or ung in seen:
            continue
        seen.add(ung)
        selected.append(s)
        if len(selected) >= N:
            break

    # if include_target=False, we still may have 0 entries; handle that gracefully
    if not include_target:
        # re-run fill without target (already done) — nothing else needed
        pass

    # --- writes ---
    if out_aligned:
        msa = MultipleSeqAlignment([s["rec"] for s in selected])
        AlignIO.write(msa, out_aligned, "fasta")

    if out_fasta:
        out_records = []
        for s in selected:
            rec = s["rec"]
            ung = _ungap(str(rec.seq))
            pid_pct = s["pid"] * 100.0
            cov_pct = s["cov"] * 100.0
            header = (
                f"{rec.id}"
                f"|pid={pid_pct:.2f}%|cov={cov_pct:.2f}%|len={len(ung)}"
                f"|target={'1' if s['is_target'] else '0'}"
            )
            out_records.append(SeqRecord(Seq(ung), id=header, description=""))
        SeqIO.write(out_records, out_fasta, "fasta")

    # --- summary ---
    if selected:
        present_target = any(s["is_target"] for s in selected)
        print(f"[topN_from_msa] include_target={include_target}  target_present={present_target}")
        print(f"[topN_from_msa] Target: {target_rec.id}  (selected first)")
        best = max(selected, key=lambda s: (s["pid"], s["cov"], s["gap_lightness"]))
        print(f"[topN_from_msa] Selected {len(selected)} (N={N}, min_cov={min_coverage:.2f}); "
              f"best hit: {best['rec'].id} pid={best['pid']:.3f} cov={best['cov']:.3f} "
              f"{'(target)' if best['is_target'] else ''}")

    return [(s["rec"].id, s["pid"], s["cov"], len(_ungap(str(s["rec"].seq))), s["is_target"]) for s in selected]

# Run
topN_from_msa(
    "PF10613.full.fasta",
    target_hint="NMDZ1_HUMAN",                 
    out_fasta="PF10613_top100.ungapped.fasta",
    out_aligned="PF10613_top100.aligned.fasta",
    N=100,
    min_coverage=0.90,
    include_target=True
)
print("Wrote: PF10613_top100.aligned.fasta and PF10613_top100.ungapped.fasta")

[topN_from_msa] include_target=True  target_present=True
[topN_from_msa] Target: NMDZ1_HUMAN/444-546  (selected first)
[topN_from_msa] Selected 100 (N=100, min_cov=0.90); best hit: NMDZ1_HUMAN/444-546 pid=1.000 cov=1.000 (target)
Wrote: PF10613_top100.aligned.fasta and PF10613_top100.ungapped.fasta


# Step 4: Clean & Annotate file with Selected Data

In [104]:
# Clean aligned MSA and write headers as: ENTRY|common_name|PID:xx.xx%
# - Computes %ID vs target over target non-gap columns
# - Resolves species using FULL Pfam FASTA first, then speclist/fallback
# - Outputs an aligned FASTA (same columns), just with nice headers

import os, re, collections
from typing import Dict, Optional, Tuple
from Bio import AlignIO, SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

# ---------------- Config ----------------
IN_ALIGNED   = "PF10613_top100.aligned.fasta"
OUT_CLEAN    = "PF10613_top100.clean.fasta"
FULL_FASTA   = "PF10613.full.fasta"            # for species lookup
SPECLIST_TXT = "Uniprot_Species_list.txt"

MNEMONIC_FALLBACK: Dict[str, str] = {
    "9TELE": "fish",
    "9CHAR": "shorebird",
    "9AMPH": "amphibian",
    "9SAUR": "reptile",
    "9PASS": "passerine",
    "9AVES": "bird",
    "9HYME": "hymenoptera",
    "9BILA": "bilateria",
    "9CORV": "corvid",
    "9EUCA": "eukaryote",
    "9BRAS": "crucifers"
}
GAPS = {"-", "."}

# -------------- Helpers --------------
def to_snake_lower(s: str) -> str:
    s = s.strip().replace("-", " ")
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", "_", s)
    return s.lower()

def load_speclist_real_codes(path: str) -> Dict[str, Dict[str, Optional[str]]]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Speclist file not found: {path}")
    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        lines = fh.readlines()
    start = next((i for i,l in enumerate(lines) if l.strip()=="(1) Real organism codes"), None)
    if start is None:
        raise ValueError("Could not find '(1) Real organism codes'")
    end = next((i for i in range(start+1,len(lines)) if lines[i].strip().startswith("(2)")), len(lines))
    entry_pat = re.compile(r"^([A-Z0-9]{2,6})\s+[A-Z]\s+\d+:\s+N=(.*)$")
    cont_pat  = re.compile(r"^\s+[A-Z]=.*$")
    table = {}
    i = start
    while i < end:
        m = entry_pat.match(lines[i].rstrip("\n"))
        if m:
            code, sci = m.groups()
            code = code.upper()
            table[code] = {"scientific": sci.strip(), "common": None}
            i += 1
            while i < end and cont_pat.match(lines[i]):
                cont = lines[i].strip()
                if cont.startswith("C=") and table[code]["common"] is None:
                    table[code]["common"] = cont[2:].strip()
                i += 1
            continue
        i += 1
    return table

def build_species_lookup_from_full(full_fasta: str) -> Dict[str, str]:
    lookup = {}
    for rec in SeqIO.parse(full_fasta, "fasta"):
        parts = rec.id.split("|")
        if len(parts) >= 2 and parts[1] and "=" not in parts[1]:
            base_id = parts[0]
            lookup[base_id] = to_snake_lower(parts[1])
    return lookup

# allow digit-leading mnemonics (e.g., 9TELE)
_RE_MNEMO_SPTR = re.compile(r"^(?:sp|tr)\|[A-Z0-9]+\|[A-Za-z0-9]+_([A-Za-z0-9]{2,6})(?:$|/|\|)")
_RE_MNEMO_ANY  = re.compile(r"(?:^|[|])([A-Za-z0-9]+)_([A-Za-z0-9]{2,6})(?:$|/|\|)")
_RE_SPAN_SPTR  = re.compile(r"^(?:sp|tr)\|([A-Za-z0-9]+)\|([A-Za-z0-9]+)_[A-Za-z0-9]+(?:/(\d+-\d+))?$")
_RE_SPAN_ANY   = re.compile(r"^([A-Za-z0-9]+)_[A-Za-z0-9]+(?:/(\d+-\d+))?$")

def extract_mnemonic(header_id: str) -> Optional[str]:
    m = _RE_MNEMO_SPTR.search(header_id)
    if m: return m.group(1).upper()
    m = _RE_MNEMO_ANY.search(header_id)
    if m: return m.group(2).upper()
    return None

def entry_core(base: str) -> str:
    m = _RE_SPAN_SPTR.match(base)
    if m:
        _acc, entry, _span = m.groups()
        return entry
    m = _RE_SPAN_ANY.match(base)
    if m:
        entry, _span = m.groups()
        return entry
    if "_" in base:
        return base.split("_", 1)[0]
    return base

def split_base_and_trailing(raw_id: str) -> Tuple[str, str]:
    if "|" in raw_id:
        b, t = raw_id.split("|", 1)
        return b, "|" + t
    return raw_id, ""

def common_name_for_mnemonic(code: Optional[str],
                             speclist: Dict[str, Dict[str, Optional[str]]],
                             fallback: Dict[str, str]) -> Optional[str]:
    if not code:
        return None
    code = code.upper()
    if code in speclist:
        nm = speclist[code].get("common") or speclist[code].get("scientific")
        return to_snake_lower(nm) if nm else None
    if code in fallback:
        return to_snake_lower(fallback[code])
    return None

def _find_target_index(aln, target_hint: Optional[str] = None) -> int:
    prefs = []
    if target_hint:
        prefs.append(str(target_hint).upper())
    prefs += ["NMDZ1_HUMAN", "Q05586"]
    for pref in prefs:
        for i, rec in enumerate(aln):
            txt = " ".join([rec.id, getattr(rec, "name",""), getattr(rec,"description","")]).upper()
            if pref in txt:
                return i
    return 0

def _pid_vs_target(target_seq, other_seq) -> float:
    t = str(target_seq); o = str(other_seq)
    assert len(t) == len(o)
    idx = [i for i,a in enumerate(t) if a not in GAPS]
    if not idx: return 0.0
    matches = covered = 0
    for i in idx:
        if o[i] not in GAPS:
            covered += 1
            if t[i].upper() == o[i].upper():
                matches += 1
    return (matches / covered) if covered else 0.0

# -------------- Main --------------
def clean_with_pid(in_aligned: str,
                   out_clean: str,
                   full_fasta_for_lookup: str,
                   speclist_path: str,
                   target_hint: Optional[str] = "Q05586") -> None:
    aln = AlignIO.read(in_aligned, "fasta")
    N, L = len(aln), len(aln[0].seq) if aln else (0, 0)

    speclist = load_speclist_real_codes(speclist_path)
    fallback = {k.upper(): to_snake_lower(v) for k, v in MNEMONIC_FALLBACK.items()}
    species_lookup = build_species_lookup_from_full(full_fasta_for_lookup)

    t_idx = _find_target_index(aln, target_hint=target_hint)
    target = aln[t_idx]

    out_records = []
    species_counts = collections.Counter()
    unknown = []

    for rec in aln:
        base, _trailing = split_base_and_trailing(rec.id)
        entry = entry_core(base)
        mnemo = extract_mnemonic(rec.id)

        # species priority: FULL lookup -> speclist/fallback -> mnemonic -> unknown
        sp = species_lookup.get(base) or species_lookup.get(base.split("/",1)[0])
        if not sp:
            sp = common_name_for_mnemonic(mnemo, speclist, fallback)
        if not sp and mnemo:
            sp = to_snake_lower(mnemo)
        if not sp:
            sp = "unknown"
            unknown.append(rec.id)

        pid_pct = _pid_vs_target(target.seq, rec.seq) * 100.0
        new_id = f"{entry}|{sp}|PID:{pid_pct:.2f}%"

        out_records.append(SeqRecord(Seq(str(rec.seq)), id=new_id, description=""))
        species_counts[sp] += 1

    SeqIO.write(out_records, out_clean, "fasta")
    print(f"[annotated] Wrote {out_clean}  (N={len(out_records)}, L={L})")
    print("[Taxonomic coverage] top species tokens:")
    for sp, n in species_counts.most_common(15):
        print(f"  {sp:30s} {n:6d}")
    if unknown:
        print(f"[warn] species unresolved for {len(unknown)} ids:")
        for rid in unknown:
            print("  -", rid)

# ---- Run ----
clean_with_pid(
    IN_ALIGNED,
    OUT_CLEAN,
    FULL_FASTA,
    SPECLIST_TXT,
    target_hint="NMDZ1_HUMAN"   # or "Q05586"
)

[annotated] Wrote PF10613_top100.clean.fasta  (N=100, L=780)
[Taxonomic coverage] top species tokens:
  fish                               14
  goldfish                            4
  dianchi_golden_line_fish            3
  atlantic_cod                        2
  african_clawed_frog                 2
  mangrove_killifish                  2
  zebrafish                           2
  kaup_s_arrowtooth_eel               2
  european_freshwater_eel             2
  pygmy_chimpanzee                    2
  yangtze_river_dolphin               2
  guppy                               2
  climbing_perch                      2
  human                               1
  bovine                              1


In [105]:
# Print file head to get a look at the file
from Bio import SeqIO

records = list(SeqIO.parse("PF10613_top100.clean.fasta", "fasta"))
for r in records[:10]:
    print(r.id)

NMDZ1|human|PID:100.00%
A0AAA9RTM8|bovine|PID:99.03%
A0A2Y9SXP4|sperm_whale|PID:99.03%
A0A7J7FGE5|south_central_black_rhinoceros|PID:99.03%
A0A2K5UT02|crab_eating_macaque|PID:99.02%
A0A2I3HC39|northern_white_cheeked_gibbon|PID:98.96%
A0AAD1T365|western_spadefoot_toad|PID:96.00%
A0A8C5AY71|atlantic_cod|PID:95.74%
F7CS95|western_clawed_frog|PID:95.10%
A0A6P6K8F5|goldfish|PID:95.05%


Run in Command:<br>
muscle -align PF10613_top100.clean.fasta -output PF10613_top100.clean.aln

# Question 1: Taxonomy & Redundancy

In [106]:
# Henikoff weights
"""
henikoff_weights() is a method (Henikoff & Henikoff, 1994) to reduce redundancy bias 
in a multiple sequence alignment (MSA).
When you have many very similar sequences (e.g., dozens of mammals), they can overwhelm 
less-represented clades (like a single reptile or fish). Henikoff weighting corrects that by giving 
less weight to over-represented sequences and more weight to unique ones.
"""
import numpy as np, pandas as pd, re
from collections import Counter
from Bio import AlignIO

def henikoff_weights(aln):
    # Convert Biopython alignment into a NumPy array of characters.
    # Shape: nseq x ncol
    A = np.array([list(str(r.seq)) for r in aln])
    nseq, ncol = A.shape

    # Initialize a weight for each sequence (starts at 0)
    w = np.zeros(nseq)

    # Loop over each alignment column
    for j in range(ncol):
        col = A[:, j]

        # Mask to ignore gaps ('-')
        mask = col != '-'

        # Count how many unique residue types appear in this column (non-gapped only)
        syms, cnt = np.unique(col[mask], return_counts=True)
        if cnt.size == 0:
            continue  # skip fully gapped columns

        k = cnt.size              # number of residue types in this column
        look = dict(zip(syms, cnt))  # residue -> how many times it occurs

        # For each sequence, if it has a residue here, add fractional weight
        for i in range(nseq):
            if mask[i]:
                # Each residue contributes 1/(k * count_of_that_residue)
                # -> Rare residues (low count) and diverse columns (large k) increase weight
                w[i] += 1.0 / (k * look[col[i]])

    # Normalize weights so they sum to 1.0
    w /= w.sum()
    return w

def aa_fraction(seq):
    s = str(seq).upper()
    L = sum(a not in GAPS for a in s)
    return L

aln = AlignIO.read("PF10613_top100.clean.aln", "fasta")
GAPS = {"-","."}

# --- Henikoff weights & Neff/L (you already have henikoff_weights) ---
W = henikoff_weights(aln)                   # sums to 1
Neff = 1.0 / (W**2).sum()
L = aln.get_alignment_length()
print(f"[Redundancy] Neff={Neff:.1f}, L={L}, Neff/L={Neff/L:.2f}")

# Rough heuristic:
#   Neff/L < 0.5  shallow (weak coevolution)
#   0.5–2.0       moderate
#   >2.0          strong (good for MI/DCA)


[Redundancy] Neff=31.9, L=153, Neff/L=0.21


In [107]:
# Taxonomy & Redundancy Summary for PF10613_top100.clean.aln 
import math
import numpy as np
import pandas as pd
from collections import Counter
from itertools import combinations
from Bio import AlignIO

# ------------- helpers -------------
def parse_species_from_id(rec_id: str) -> str:
    """F
    Expect headers like: ENTRY|species_token|PID:xx.xx%
    Falls back gracefully if not present.
    """
    parts = rec_id.split("|")
    if len(parts) >= 2 and parts[1]:
        return parts[1]
    return "unknown"

def seq_nongap_fraction(seq_str: str, gaps=set("-.")) -> float:
    s = seq_str
    if not s: return 0.0
    nongap = sum(c not in gaps for c in s)
    return nongap / len(s)

def pid_ignore_gaps(a: np.ndarray, b: np.ndarray, gaps=set("-.")) -> float:
    """
    Pairwise %identity ignoring positions where either seq has a gap.
    a, b are 1D dtype '<U1' arrays (characters). Returns fraction 0..1.
    """
    mask = (a != "-") & (a != ".") & (b != "-") & (b != ".")
    m = mask.sum()
    if m == 0: 
        return 0.0
    return (a[mask] == b[mask]).mean()

def neff_by_threshold(A: np.ndarray, theta: float = 0.8) -> float:
    """
    Sequence-reweighting Neff using identity threshold theta.
    w_i = 1 / |{ j : PID(i,j) >= theta }|
    Neff = sum_i w_i
    """
    N = A.shape[0]
    Wt = np.zeros(N, dtype=float)
    for i in range(N):
        cnt = 0
        for j in range(N):
            cnt += (pid_ignore_gaps(A[i], A[j]) >= theta)
        Wt[i] = 1.0 / max(1, cnt)
    return Wt.sum(), Wt

def column_gap_fraction(A: np.ndarray, gaps=set("-.")) -> np.ndarray:
    """Fraction of gaps per column, 0..1"""
    return ((A == "-") | (A == ".")).mean(axis=0)

def shannon_diversity(counts: Counter) -> float:
    """Shannon entropy H (nat) over species counts."""
    total = sum(counts.values())
    if total == 0: return 0.0
    H = 0.0
    for n in counts.values():
        p = n / total
        H -= p * math.log(p)
    return H

# ------------- compute tables -------------
# Parse alignment & matrix
aln = AlignIO.read("PF10613_top100.clean.aln", "fasta")
names = [r.id for r in aln]
seqs  = [str(r.seq).upper() for r in aln]
N     = len(seqs)
L     = len(seqs[0]) if N else 0
A     = np.array([list(s) for s in seqs], dtype="<U1")

# Species parsing (from your cleaned headers)
species = [parse_species_from_id(r.id) for r in aln]
species_counts = Counter(species)
unique_species = len(species_counts)

# Per-sequence coverage (non-gap fraction)
seq_cov = np.array([seq_nongap_fraction(s, GAPS) for s in seqs])

# Pairwise identity summary (ignore gaps)
# (For N=100 this O(N^2 L) loop is totally fine.)
pids = []
for i in range(N):
    ai = A[i]
    for j in range(i+1, N):
        pids.append(pid_ignore_gaps(ai, A[j]))
pids = np.array(pids) if pids else np.array([0.0])


# Neff via threshold-based reweighting (common reporting)
Neff80, W80 = neff_by_threshold(A, theta=0.80)
Neff90, W90 = neff_by_threshold(A, theta=0.90)

# Column gap profile
col_gap = column_gap_fraction(A, GAPS)
cols_mostly_gapped = int((col_gap >= 0.5).sum())
cols_well_covered  = int((col_gap <= 0.2).sum())

# Species diversity metrics
H = shannon_diversity(species_counts)           # nat
H_max = math.log(unique_species) if unique_species > 0 else 1.0
evenness = (H / H_max) if H_max > 0 else 0.0    # Pielou's J, 0..1

# ------------- summary dataframe -------------
summary = pd.DataFrame({
    "metric": [
        "N sequences",
        "Alignment length (L)",
        "Mean seq coverage",
        "Median seq coverage",
        "Median pairwise %ID",
        "IQR pairwise %ID",
        "Henikoff Neff",
        "Henikoff Neff/L",
        "Neff@0.80",
        "Neff@0.80 / L",
        "Neff@0.90",
        "Neff@0.90 / L",
        "Unique species (tokens)",
        "Shannon diversity H (nat)",
        "Evenness (H / log(S))",
        "Cols ≥50% gaps",
        "Cols ≤20% gaps",
    ],
    "value": [
        N,
        L,
        float(seq_cov.mean()),
        float(np.median(seq_cov)),
        float(np.median(pids)*100.0),
        float((np.percentile(pids, 75)-np.percentile(pids,25))*100.0),
        float(Neff),
        float(Neff / L if L else 0.0),
        float(Neff80),
        float(Neff80 / L if L else 0.0),
        float(Neff90),
        float(Neff90 / L if L else 0.0),
        unique_species,
        float(H),
        float(evenness),
        cols_mostly_gapped,
        cols_well_covered,
    ],
    "notes": [
        "",
        "",
        "Fraction non-gap per sequence (mean)",
        "Median across sequences",
        "Median across all sequence pairs (×100 for %)",
        "IQR across pairs (×100 for %)",
        "From Henikoff weights (given)",
        "Rule of thumb: ≥2 is solid for DCA",
        "Seq-reweighting with θ=0.80",
        "",
        "Seq-reweighting with θ=0.90",
        "",
        "Count of distinct species tokens parsed from headers",
        "Higher = more diverse",
        "0..1, higher = more even",
        "High-gap columns (may prune or mask)",
        "Well-covered columns (good signal)",
    ]
})

# Nice formatting
# --- Safer pretty-format for display ---
summary_display = summary.copy()
summary_display["formatted"] = summary_display["value"]

pct_cols = {
    "Mean seq coverage",
    "Median seq coverage",
    "Median pairwise %ID",
    "IQR pairwise %ID",
    "Henikoff Neff/L",
    "Neff@0.80 / L",
    "Neff@0.90 / L"
}

def _fmt_value(row):
    val = row["value"]
    if row["metric"] in pct_cols:
        if val <= 1.0:
            return f"{val*100:.2f}%"
        return f"{val:.2f}%"
    if isinstance(val, float):
        return f"{val:.3f}"
    return val

summary_display["formatted"] = summary_display.apply(_fmt_value, axis=1)


# ------------- detailed tables (optional) -------------
# Species counts table
species_table = pd.DataFrame(
    [{"species": s, "count": n, "fraction": n/N} for s,n in species_counts.most_common()]
)
# Per-sequence stats table
per_seq_table = pd.DataFrame({
    "id": names,
    "species": species,
    "coverage": seq_cov,
    "weight_henikoff": W,
    "weight@0.80": W80,
    "weight@0.90": W90
}).sort_values("coverage", ascending=False)

# ------------- print/save -------------
pd.set_option("display.max_rows", 200)
print("\n=== Taxonomy & Redundancy Summary ===")
print(summary_display.to_string(index=False))

print("\n=== Top species ===")
print(species_table.head(15).to_string(index=False))

# Save CSVs
# summary.to_csv("PF10613_top100.stats.summary.csv", index=False)
# species_table.to_csv("PF10613_top100.stats.species_counts.csv", index=False)
# per_seq_table.to_csv("PF10613_top100.stats.per_sequence.csv", index=False)
# print("\n[write] PF10613_top100.stats.summary.csv")
# print("[write] PF10613_top100.stats.species_counts.csv")
# print("[write] PF10613_top100.stats.per_sequence.csv")



=== Taxonomy & Redundancy Summary ===
                   metric      value                                                notes formatted
              N sequences 100.000000                                                        100.000
     Alignment length (L) 153.000000                                                        153.000
        Mean seq coverage   0.669281                 Fraction non-gap per sequence (mean)    66.93%
      Median seq coverage   0.656863                              Median across sequences    65.69%
      Median pairwise %ID  93.877551        Median across all sequence pairs (×100 for %)    93.88%
         IQR pairwise %ID   5.208333                        IQR across pairs (×100 for %)     5.21%
            Henikoff Neff  31.944277                        From Henikoff weights (given)    31.944
          Henikoff Neff/L   0.208786                   Rule of thumb: ≥2 is solid for DCA    20.88%
                Neff@0.80   1.000202                         

In [164]:
# Convert to phylip-relaxed
from Bio import AlignIO

# The format string should be lowercase and not include the path
# The fourth parameter should be just the format name: "phylip-relaxed"
AlignIO.convert("PF10613_top100.clean.aln", "fasta", 
                "PF10613_top100.phy", "phylip-relaxed")

1

In [165]:
# Read in phylip-relaxed
aln = AlignIO.read("PF10613_top100.phy", "phylip-relaxed")
print("Sequences:", len(aln))
print("Alignment length:", aln.get_alignment_length())

Sequences: 100
Alignment length: 153


In [168]:
# Produce PhyloTree
import re
from Bio import AlignIO, Phylo
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor

#aln = AlignIO.read(aln_path, "phylip-relaxed")

# --------- Build tree (NJ with BLOSUM62 distances) ---------
calc = DistanceCalculator("blosum62")
dm = calc.get_distance(aln)
constructor = DistanceTreeConstructor(calc, "nj")
tree = constructor.build_tree(aln)

# --------- Save & quick preview ---------
out_newick = "PF10613_seed.labeled.nwk"
Phylo.write(tree, out_newick, "newick")
print(f"Wrote {out_newick}")
Phylo.draw_ascii(tree)

Wrote PF10613_seed.labeled.nwk
  , F1R366|zebrafish|PID|90.82%
 ,|
 ||  , A0A8C4F2U9|european_seabass|PID|87.38%
 ||__|
 |   | A0A7N8XCT4|fish|PID|88.00%
 |
 |__ A0A665VYX8|live_sharksucker|PID|88.12%
 |
 |_____ A0A672S9Y3|dianchi_golden_line_fish|P...
 |
 | , A0A672SAR7|dianchi_golden_line_fish|P...
 |,|
 |||  ______ A0A6F9A562|fish|PID|88.78%
 |||_|
 ||  |, A0A7N6FAN3|climbing_perch|PID|89.69%
 ||  ||
 ||   |________ A0A3Q3GAP3|mangrove_killifish|PID|92.16%
 ||
 || A0A673LP77|fish|PID|90.72%
 |
 | , A0A667YQ01|fish|PID|88.89%
 |,|
 |||_ A0A8M1HHT2|siamese_fighting_fish|PID|...
 ||
 ||, A0A3P9N449|guppy|PID|88.89%
 |||
 ||| A0A8C2X404|lumpsucker|PID|89.32%
 |,|
 ||, A0A7N6C393|climbing_perch|PID|87.88%
 |||
 ||, A0A9Y4TZR2|fish|PID|90.29%
 |||
 ||| A0A8C7WTV9|fish|PID|88.00%
 ||
 ||, A0A669BP57|nile_tilapia|PID|90.20%
 |,|
 ||| A0A6G1PYJ9|fish|PID|88.89%
 ||
 || A0A3Q4I9W9|fairy_cichlid|PID|89.00%
 ||
 || , A0A672HAF5|jewelled_blenny|PID|89.90%
 ||,|
 ||||  __ A0AA47MAT5|benguela_hake

In [119]:
# Nearest Neighbor to Target
from Bio import Phylo
from collections import namedtuple

import re
from collections import namedtuple
from Bio import Phylo

def parse_leaf(name: str):
    """
    Accepts both formats:
      ID|species|PID|90.82%
      ID|species|pid=90.82%
    Returns (id_upper, species_lower, pid_float or None).
    """
    parts = name.split("|")
    leaf_id = parts[0].upper() if parts else ""
    species = parts[1].lower() if len(parts) > 1 else ""
    pid = None
    # try to parse a percent from any of the trailing parts
    for p in parts[2:]:
        m = re.search(r"(\d+(?:\.\d+)?)\s*%$", p)
        if m:
            pid = float(m.group(1))
            break
        m = re.search(r"pid\s*=\s*(\d+(?:\.\d+)?)\s*%?$", p, re.I)
        if m:
            pid = float(m.group(1))
            break
    return leaf_id, species, pid


# ---- find the target clade by ID or species token (case-insensitive) ----
target_clade = None
for cl in tree.get_terminals():
    leaf_id, species, pid = parse_leaf(cl.name)
    if (
        leaf_id.startswith("NMDZ1") or
        leaf_id.startswith("Q05586") or
        species in {"human", "homo_sapiens"}
    ):
        target_clade = cl
        break

if target_clade is None:
    leaves_preview = [cl.name for cl in tree.get_terminals()[:8]]
    raise ValueError(
        "Target not found among leaf labels with current matching rules.\n"
        f"Sample leaves: {leaves_preview}"
    )

# ---- nearest/farthest neighbors by patristic distance ----
Tip = namedtuple("Tip", "name dist")
tips = []
for cl in tree.get_terminals():
    if cl is target_clade:
        continue
    d = float(tree.distance(target_clade, cl))
    tips.append(Tip(cl.name, d))

closest  = sorted(tips, key=lambda t: t.dist)[:15]
farthest = sorted(tips, key=lambda t: t.dist, reverse=True)[:15]

print("\n[Nearest to target]")
for t in closest:
    print(f"{t.dist:7.3f}  {t.name}")

print("\n[Farthest from target]")
for t in farthest:
    print(f"{t.dist:7.3f}  {t.name}")


[Nearest to target]
 -0.000  A0A340XI92|yangtze_river_dolphin|PID|90.29%
  0.000  A0A340XBX0|yangtze_river_dolphin|PID|90.29%
  0.001  A0AAA9RTM8|bovine|PID|99.03%
  0.009  A0A1S3G7X9|ord_s_kangaroo_rat|PID|90.29%
  0.015  A0A8C4MD47|donkey|PID|90.29%
  0.015  A0A7J8IKA1|egyptian_fruit_bat|PID|90.29%
  0.015  A0A1S2ZMA1|western_european_hedgehog|PID|90.29%
  0.019  A0A7J7FGE5|south_central_black_rhinoceros|PID|99.03%
  0.026  A0A2K5UT02|crab_eating_macaque|PID|99.02%
  0.028  A0A2Y9SXP4|sperm_whale|PID|99.03%
  0.038  A0A2I3HC39|northern_white_cheeked_gibbon|PID|98.96%
  0.042  F7E6L8|duckbill_platypus|PID|89.32%
  0.043  W5LWF6|spotted_gar|PID|91.84%
  0.045  A0A8B9X1W9|wild_yak|PID|91.26%
  0.046  A0AAD1T365|western_spadefoot_toad|PID|96.00%

[Farthest from target]
  0.154  A0A8C5BAB2|atlantic_cod|PID|94.68%
  0.146  A0A672S9Z2|dianchi_golden_line_fish|PID|89.22%
  0.124  A0AAD9BZW0|patagonian_toothfish|PID|89.69%
  0.117  A0A4W3J4J6|ghost_shark|PID|87.63%
  0.116  A0A2R9AFT8|pygmy_

# Question 2: Which regions are highly conserved vs hypervariable? Do conserved positions cluster near known functional sites (ligand-binding, gating, interfaces)?

In [134]:
import numpy as np, pandas as pd
from Bio import AlignIO
from math import log2

aln_path = "PF10613_top100.clean.aln"
GAPS = {"-", "."}

# --- Load alignment ---
aln = AlignIO.read(aln_path, "fasta")
A = np.array([list(str(rec.seq)) for rec in aln])
nseq, ncol = A.shape

# --- Shannon entropy per column ---
def shannon_entropy(column):
    """Compute Shannon entropy (base 2) for a single column."""
    symbols, counts = np.unique([c for c in column if c not in GAPS], return_counts=True)
    if len(symbols) == 0:
        return np.nan
    freqs = counts / counts.sum()
    return -np.sum(freqs * np.log2(freqs))

entropies = np.array([shannon_entropy(A[:, j]) for j in range(ncol)])

# --- Fraction of the majority residue per column ---
major_fracs = []
major_residues = []
for j in range(ncol):
    col = [c for c in A[:, j] if c not in GAPS]
    if not col:
        major_fracs.append(np.nan)
        major_residues.append("-")
        continue
    syms, counts = np.unique(col, return_counts=True)
    i = np.argmax(counts)
    major_residues.append(syms[i])
    major_fracs.append(counts[i] / len(col))

# --- Normalize entropy (for visual bins) ---
Hmax = log2(20)  # max possible for 20 aa types
Hnorm = entropies / Hmax

# --- Assign qualitative category ---
cats = pd.cut(
    Hnorm,
    bins=[-0.01, 0.15, 0.45, 1.00],
    labels=["Conserved", "Moderate", "Variable"]
)

# --- Combine into a summary table ---
summary = pd.DataFrame({
    "Position": np.arange(1, ncol + 1),
    "Major_AA": major_residues,
    "Major_Freq": np.round(major_fracs, 3),
    "Entropy": np.round(entropies, 3),
    "NormEntropy": np.round(Hnorm, 3),
    "Category": cats
})

# --- Basic stats for reporting ---
print(f"[Conservation summary]")
print(f"Total columns: {ncol}")
print(f"Mean entropy: {np.nanmean(entropies):.3f}")
print(f"Highly conserved (<0.3 bits): {(Hnorm < 0.15).sum()}")
print(f"Hypervariable (>2.0 bits): {(entropies > 2.0).sum()}")

# --- Display first few and also aggregated counts ---
display_cols = ["Position", "Major_AA", "Major_Freq", "Entropy", "Category"]
print(summary[display_cols].head(10).to_string(index=False))

category_counts = summary["Category"].value_counts().sort_index()
print("\n[Category distribution]")
print(category_counts)


[Conservation summary]
Total columns: 153
Mean entropy: 0.363
Highly conserved (<0.3 bits): 110
Hypervariable (>2.0 bits): 3
 Position Major_AA  Major_Freq  Entropy  Category
        1        T       0.600    1.371  Moderate
        2        R       0.667    0.918  Moderate
        3        L       0.750    0.811  Moderate
        4        K       0.750    0.811  Moderate
        5        I       0.750    0.811  Moderate
        6        V       0.750    0.811  Moderate
        7        T       1.000   -0.000 Conserved
        8        I       0.750    0.811  Moderate
        9        H       0.750    0.811  Moderate
       10        Q       0.750    0.811  Moderate

[Category distribution]
Category
Conserved    110
Moderate      39
Variable       4
Name: count, dtype: int64


<b>Note:</b><br>
The domain is highly conserved overall (71.9%, 110/153), consistent with its critical role<br>
Only ~3% (4/153) positions are variable<br>
Moderate sites (25.5%, 39/153) might correspond to surface residues, interface patches, or tolerated substitutions between taxa (fish vs mammals)

In [142]:
import pandas as pd
import numpy as np

# Use your summary DF from earlier
df = summary.copy()

# --- Parameters ---
window = 5
conserved_bits   = 0.30          # bits (very strict)
variable_bits    = 1.80          # was 2.0; relax a bit
use_normalized   = False         # set True to use Hnorm thresholds instead
conserved_hnorm  = 0.15          # ~= 0.3 / log2(20)
variable_hnorm   = 0.45          # ~= 1.95 / log2(20)
topK_fallback    = 5             # ensure we show something even if threshold empty

# --- Rolling mean entropy (be permissive at edges) ---
df["Entropy_roll"] = df["Entropy"].rolling(window, center=True, min_periods=1).mean()

# --- Choose thresholds (bits vs normalized) ---
if use_normalized:
    key = "NormEntropy"
    cons_mask = df["NormEntropy"].le(conserved_hnorm)
    var_mask  = df["NormEntropy"].ge(variable_hnorm)
    cons_mask_roll = df["Entropy_roll"].div(np.log2(20)).le(conserved_hnorm)
    var_mask_roll  = df["Entropy_roll"].div(np.log2(20)).ge(variable_hnorm)
else:
    key = "Entropy"
    cons_mask = df["Entropy"].le(conserved_bits)
    var_mask  = df["Entropy"].ge(variable_bits)
    cons_mask_roll = df["Entropy_roll"].le(conserved_bits)
    var_mask_roll  = df["Entropy_roll"].ge(variable_bits)

# --- Pick columns by rolling mean (preferred) ---
cons_cols = df.loc[cons_mask_roll, ["Position", "Entropy", "Entropy_roll"]]
var_cols  = df.loc[var_mask_roll,  ["Position", "Entropy", "Entropy_roll"]]

# Fallbacks if empty: use per-column thresholds
if var_cols.empty:
    var_cols = df.loc[var_mask, ["Position", "Entropy", "Entropy_roll"]]

# Final fallback: top-K highest rolling entropy windows
if var_cols.empty and topK_fallback:
    var_cols = df.nlargest(topK_fallback, "Entropy_roll")[["Position", "Entropy", "Entropy_roll"]].sort_values("Position")

# --- Collapse contiguous positions into regions ---
def collapse_regions(df_subset):
    if df_subset.empty:
        return pd.DataFrame(columns=["Start","End","Length","Mean_Entropy","Mean_Roll"])
    pos = df_subset["Position"].to_numpy()
    ent = df_subset["Entropy"].to_numpy()
    rol = df_subset["Entropy_roll"].to_numpy()
    regions = []
    start = pos[0]; prev = pos[0]
    ent_buf = [ent[0]]; rol_buf = [rol[0]]
    for p, e, r in zip(pos[1:], ent[1:], rol[1:]):
        if p == prev + 1:
            ent_buf.append(e); rol_buf.append(r)
        else:
            regions.append((start, prev, prev-start+1, float(np.mean(ent_buf)), float(np.mean(rol_buf))))
            start = p; ent_buf = [e]; rol_buf = [r]
        prev = p
    regions.append((start, prev, prev-start+1, float(np.mean(ent_buf)), float(np.mean(rol_buf))))
    return pd.DataFrame(regions, columns=["Start","End","Length","Mean_Entropy","Mean_Roll"])

cons_regions = collapse_regions(cons_cols)
var_regions  = collapse_regions(var_cols)

# --- Build pretty tables ---
def make_table(reg_df, note):
    if reg_df.empty:
        return pd.DataFrame(columns=["Region (Alignment Pos.)","Length","Mean Entropy (bits)","Mean (rolling)","Notes"])
    reg_df = reg_df.copy()
    reg_df["Region (Alignment Pos.)"] = reg_df.apply(lambda x: f"{int(x.Start)}–{int(x.End)}", axis=1)
    reg_df["Mean Entropy (bits)"] = reg_df["Mean_Entropy"].round(3)
    reg_df["Mean (rolling)"] = reg_df["Mean_Roll"].round(3)
    reg_df["Notes"] = note
    cols = ["Region (Alignment Pos.)","Length","Mean Entropy (bits)","Mean (rolling)","Notes"]
    return reg_df[cols].sort_values(by=["Length","Mean (rolling)"], ascending=[False, True])

table_conserved = make_table(cons_regions, "Highly conserved")
table_variable  = make_table(var_regions,  "Hypervariable")

print("Most Conserved Regions")
display(table_conserved.head(10))

print("Least Conserved Regions")
display(table_variable.head(10))

# Diagnostics so you can see what's happening
print("\n[diag] counts:",
      f"conserved_roll={len(cons_cols)}",
      f"variable_roll={len(var_cols)} (after fallbacks)")


Most Conserved Regions


,Region (Alignment Pos.),Length,Mean Entropy (bits),Mean (rolling),Notes
5,117–151,35,0.020,0.038,Highly conserved
4,79–113,35,0.028,0.040,Highly conserved
3,60–70,11,0.156,0.154,Highly conserved
2,40–43,4,0.000,0.000,Highly conserved
0,13–15,3,0.000,0.161,Highly conserved
1,31–31,1,0.918,0.184,Highly conserved


Least Conserved Regions


,Region (Alignment Pos.),Length,Mean Entropy (bits),Mean (rolling),Notes
3,73–73,1,1.958,0.620,Hypervariable
0,26–26,1,2.170,1.131,Hypervariable
2,52–52,1,2.298,1.335,Hypervariable
1,47–47,1,2.666,1.399,Hypervariable



[diag] counts: conserved_roll=89 variable_roll=4 (after fallbacks)


In [144]:
# MSA Visulaization
from collections import Counter
from IPython.display import HTML, display
import html

# --- Load your alignment ---
aln = AlignIO.read("PF10613_top100.clean.aln", "fasta")  

# --- Helpers ---
KD = dict(A=1.8,R=-4.5,N=-3.5,D=-3.5,C=2.5,Q=-3.5,E=-3.5,G=-0.4,H=-3.2,
          I=4.5,L=3.8,K=-3.9,M=1.9,F=2.8,P=-1.6,S=-0.8,T=-0.7,W=-0.9,Y=-1.3,V=4.2)

def alignment_window(aln, start, end):
    """Return list[(id, subseq)] for 1-based inclusive window [start, end]."""
    L = aln.get_alignment_length()
    start = max(1, start); end = min(L, end)
    s0, e0 = start-1, end
    return [(rec.id, str(rec.seq[s0:e0])) for rec in aln]

def consensus_of(seqs):
    """Ungapped majority consensus over equal-length strings."""
    L = len(seqs[0])
    cons = []
    for i in range(L):
        col = [s[i] for s in seqs if s[i] != '-']
        cons.append(Counter(col).most_common(1)[0][0] if col else '-')
    return "".join(cons)

def pid_to_cons(seq, cons):
    m = sum(1 for a,b in zip(seq,cons) if a!='-' and b!='-')
    return (sum(1 for a,b in zip(seq,cons) if a==b and a!='-' and b!='-')/m) if m else 0.0

def mean_kd(seq):
    vals = [KD.get(a, 0.0) for a in seq if a != '-']
    return sum(vals)/len(vals) if vals else 0.0

def color_seq_html(seq, cons):
    """Return HTML with per-residue spans: matches tinted, mismatches accented, gaps faint."""
    out = []
    for a,b in zip(seq, cons):
        if a == '-':
            out.append('<span class="gap">-</span>')
        elif b == '-' or a != b:
            # mismatch
            out.append(f'<span class="mismatch">{html.escape(a)}</span>')
        else:
            # match to consensus
            out.append(f'<span class="match">{html.escape(a)}</span>')
    return "".join(out)

def render_window_table(aln, start, end, top_n=15, sort_by="pid"):
    rows = alignment_window(aln, start, end)
    ids, seqs = zip(*rows)
    cons = consensus_of(seqs)

    stats = []
    for rec_id, s in rows:
        stats.append({
            "id": rec_id,
            "seq_html": color_seq_html(s, cons),
            "pid": pid_to_cons(s, cons),
            "kd":  mean_kd(s)
        })
    # sort by %id (default) or hydropathy
    stats.sort(key=(lambda r: (-r["pid"], -r["kd"])) if sort_by=="pid" else (lambda r: (-r["kd"], -r["pid"])))
    stats = stats[:top_n]

    # Build HTML
    width = end - start + 1
    ruler = "".join(str((i+1)%10) for i in range(width))

    css = """
    <style>
    .msa-table {border-collapse: collapse; font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 12.5px;}
    .msa-table th, .msa-table td {border-bottom: 1px solid #e7e7e7; padding: 6px 8px; vertical-align: top;}
    .msa-table th {background: #f6f8fa; text-align: left;}
    .seq {letter-spacing: 0.5px; white-space: nowrap;}
    .cons {background:#fffbe6;}
    .ruler {color:#666;}
    .match {background: #e7f7ee;}      /* soft green for matches */
    .mismatch {background: #ffecec;}   /* soft red for mismatches */
    .gap {color:#bbb;}
    .meta {color:#333; font-variant-numeric: tabular-nums;}
    .tag {display:inline-block; background:#eef; color:#224; padding:2px 6px; border-radius:6px; margin-left:6px; font-size:11px;}
    </style>
    """

    header = f"""
    <div class="meta"><strong>Window {start}-{end}</strong>
      <span class="tag">length {width}</span>
      <span class="tag">top {len(stats)} seqs</span>
    </div>
    <table class="msa-table">
      <thead>
        <tr><th>Sequence ID</th><th>Fragment</th><th>%ID to cons</th><th>Mean KD</th></tr>
      </thead>
      <tbody>
        <tr>
          <td class="ruler">ruler</td>
          <td class="seq ruler">{ruler}</td>
          <td class="ruler"></td>
          <td class="ruler"></td>
        </tr>
        <tr>
          <td class="cons">consensus</td>
          <td class="seq cons">{color_seq_html(cons, cons)}</td>
          <td class="cons"></td>
          <td class="cons"></td>
        </tr>
    """

    rows_html = "\n".join(
        f'<tr><td>{html.escape(r["id"])}</td>'
        f'<td class="seq">{r["seq_html"]}</td>'
        f'<td class="meta">{r["pid"]*100:5.1f}%</td>'
        f'<td class="meta">{r["kd"]:5.2f}</td></tr>'
        for r in stats
    )

    footer = "</tbody></table>"
    return HTML(css + header + rows_html + footer)

"""
Most Conserved Windows
81 - 115         
121 - 155

Least Conserved Windows
1 - 50
51  - 81     
"""
# --- Render a few windows ---
print("Most Conserved windows")
display(render_window_table(aln, 81, 115, top_n=12)) # First slice
display(render_window_table(aln, 121, 155, top_n=12))
print("Least Conserved windows")
display(render_window_table(aln, 1, 50, top_n=12))
display(render_window_table(aln, 51, 95, top_n=12))


Most Conserved windows


Sequence ID,Fragment,%ID to cons,Mean KD
ruler,12345678901234567890123456789012345,,
consensus,YEVHLVADGKFGTQERVNNSNKKEWNGMMGELLSG,,
A0A2I3HC39|northern_white_cheeked_gibbon|PID:98.96%,YEVHLVADGKFGTQERVNNSNKKEWNGMMGELLSG,100.0%,-0.81
A0A8C5BAB2|atlantic_cod|PID:94.68%,YEVHLVADGKFGTQERVNNSNKKEWNGMMGELLSG,100.0%,-0.81
A0A2R9AEU9|pygmy_chimpanzee|PID:90.29%,YEVHLVADGKFGTQERVNNSNKKEWNGMMGELLSG,100.0%,-0.81
A0A2R9AFT8|pygmy_chimpanzee|PID:89.22%,YEVHLVADGKFGTQERVNNSNKKEWNGMMGELLSG,100.0%,-0.81
F7E6L8|duckbill_platypus|PID:89.32%,YEVHLVADGKFGTQERVNNSNKKEWNGMMGELLSG,100.0%,-0.81
A0A7J8IKA1|egyptian_fruit_bat|PID:90.29%,YEVHLVADGKFGTQERVNNSNKKEWNGMMGELLSG,100.0%,-0.81
A0A7N4NV16|tasmanian_devil|PID:88.35%,YEVHLVADGKFGTQERVNNSNKKEWNGMMGELLSG,100.0%,-0.81
A0A340XI92|yangtze_river_dolphin|PID:90.29%,YEVHLVADGKFGTQERVNNSNKKEWNGMMGELLSG,100.0%,-0.81


Sequence ID,Fragment,%ID to cons,Mean KD
ruler,12345678901234567890123456789012345,,
consensus,VAPLTINNERAQYIEFSKPFKYQGLTILVKKEI,,
A0A671MHF5|fish|PID:90.62%,VAPLTINNERAQYIEFSKPFKYQGLTILVKK-I,100.0%,-0.08
A0A673LP77|fish|PID:90.72%,VAPLTINNERAQYIEFSKPFKYQGLTILVKK-I,100.0%,-0.08
A0A670YRI6|eastern_brown_snake|PID:88.35%,VAPLTINNERAQYIEFSKPFKYQGLTILVKKEI,100.0%,-0.18
A0A4W3J4J6|ghost_shark|PID:87.63%,VAPLTINNERAQYIEFSKPFKYQGLTILVKKEI,100.0%,-0.18
A0A8C5BAB2|atlantic_cod|PID:94.68%,VAPLTINNERAQYIEFSKPFKYQGLTILVKKEI,100.0%,-0.18
A0A2R9AEU9|pygmy_chimpanzee|PID:90.29%,VAPLTINNERAQYIEFSKPFKYQGLTILVKKEI,100.0%,-0.18
A0AAD9BZW0|patagonian_toothfish|PID:89.69%,VAPLTINNERAQYIEFSKPFKYQGLTILVKKEI,100.0%,-0.18
F7E6L8|duckbill_platypus|PID:89.32%,VAPLTINNERAQYIEFSKPFKYQGLTILVKKEI,100.0%,-0.18


Least Conserved windows


Sequence ID,Fragment,%ID to cons,Mean KD
ruler,12345678901234567890123456789012345678901234567890,,
consensus,TRLKIVTIHQEPFVYVKCTGPNLTADGTCKPLLNSTGGLVQQVLCTSPGS,,
A0A8J0TPH2|african_clawed_frog|PID:91.00%,------------------------------------------------G-,100.0%,-0.40
A0AAD1T365|western_spadefoot_toad|PID:96.00%,------------------------------------------------G-,100.0%,-0.40
A0A8J6EI00|puerto_rican_coqui|PID:93.00%,------------------------------------------------G-,100.0%,-0.40
A0A1S3PRN4|atlantic_salmon|PID:92.00%,------------------------------------------------G-,100.0%,-0.40
A0A8C5EJF3|blunt_snouted_clingfish|PID:90.00%,------------------------------------------------G-,100.0%,-0.40
A0A8C7WTV9|fish|PID:88.00%,------------------------------------------------G-,100.0%,-0.40
A0A340XBX0|yangtze_river_dolphin|PID:90.29%,---------------------------------------------TSPGS,100.0%,-0.86
A0A340XI92|yangtze_river_dolphin|PID:90.29%,----------------------------------------------SPGS,100.0%,-0.90


Sequence ID,Fragment,%ID to cons,Mean KD
ruler,123456789012345678901234567890123456789012345,,
consensus,TPRPIVPQCCYGFCIDLLIKLAMTPTMNFTYEVHLVADGKFGTQE,,
A0A673LP77|fish|PID:90.72%,---PIVPQCCYGFCIDLLIKLAM--TMNFTYEVHLVADGKFGTQE,100.0%,0.50
A0A672S9Y3|dianchi_golden_line_fish|PID:90.53%,---PIVPQCCYGFCIDLLIKLAM--TMNFTYEVHLVADGKFGTQE,100.0%,0.50
A0A671P0S4|fish|PID:91.84%,---PIVPQCCYGFCIDLLIKLAM--TMNFTYEVHLVADGKFGTQE,100.0%,0.50
A0A667YIV4|fish|PID:90.82%,---PIVPQCCYGFCIDLLIKLAM--TMNFTYEVHLVADGKFGTQE,100.0%,0.50
A0A672SAR7|dianchi_golden_line_fish|PID:89.69%,---PIVPQCCYGFCIDLLIKLAM--TMNFTYEVHLVADGKFGTQE,100.0%,0.50
A0A7N6FAN3|climbing_perch|PID:89.69%,---PIVPQCCYGFCIDLLIKLAM--TMNFTYEVHLVADGKFGTQE,100.0%,0.50
A0A6J2QU89|frogmouth|PID:88.12%,--RPIVPQCCYGFCIDLLIKLAM--TMNFTYEVHLVADGKFGTQE,100.0%,0.38
A0A8C5EJF3|blunt_snouted_clingfish|PID:90.00%,--QPIVPQCCYGFCIDLLIKLAM--TMNFTYEVHLVADGKFGTQE,97.6%,0.40


In [162]:
# Substitution matrix gives score for amino acids replaced
from Bio import AlignIO
import numpy as np
import math

# Loads the alignment
msa = AlignIO.read("PF10613_top100.clean.aln", "fasta")

# The substitutions property of the alignment stores the number of times different residues substitute for each other
observed_frequencies = msa.substitutions

# select only amino acids with polar charged side chains
observed_frequencies = observed_frequencies.select("DEHKR")
print("Raw substitution counts:")
print("Observed substitution counts of amino acids from the MSA.")
print(observed_frequencies)

# normalize the matrix:
observed_frequencies /= np.sum(observed_frequencies)

# Summing over rows or columns gives the relative frequency of occurrence of each residue
residue_frequencies = np.sum(observed_frequencies, 0)
print("Amino acid frequencies:")
print("Marginal frequencies tell how common each residue is overall in the MSA. They’re normalized to sum to 1.")
print(residue_frequencies.format("%.4f"))

sum(residue_frequencies) == 1.0

# The expected frequency of residue pairs
expected_frequencies = np.dot(
    residue_frequencies[:, None], residue_frequencies[None, :]
)
print("Pairwise probabilities (normalized substitution frequencies):")
print(expected_frequencies.format("%.4f"))

# calculate the log-odds matrix by dividing the observed frequencies by the expected frequencies and taking the logarithm
m = np.log2(observed_frequencies / expected_frequencies)
print("Log Odds Substitution Scores:")
print("Positive scores = substitutions more frequent than expected by chance (conserved or chemically similar)")
print("Negative scores = substitutions less frequent (chemically dissimilar or rarely tolerated)")
print(m)

# This matrix can be used as the substitution matrix when performing alignments
from Bio.Align import PairwiseAligner
aligner = PairwiseAligner()
aligner.substitution_matrix = m
aligner.gap_score = -3.0
alignments = aligner.align("DEHEK", "DHHKK")
print("Aignment Snippet & Similarity Score")
print("Absolute alignment score derived by summing up substitution scores across aligned residues, based on substitution matrix.")
print(alignments[0])

score = m["D", "D"] + m["E", "H"] + m["H", "H"] + m["E", "K"] + m["K", "K"]

# Convert bits to fold-likelihood over random
fold_likelihood = 2 ** score

print(f"Alignment score: {score:.2f} bits")
print(f"Equivalent fold-likelihood over random: {fold_likelihood:.2f}x")

Raw substitution counts:
Observed substitution counts of amino acids from the MSA.
        D       E      H       K       R
D 14880.0    16.0    0.0   109.0    24.0
E    16.0 33569.0    3.0     0.5    42.5
H     0.0     3.0 5110.0     5.5    53.0
K   109.0     0.5    5.5 38819.0   291.0
R    24.0    42.5   53.0   291.0 10976.0

Amino acid frequencies:
Marginal frequencies tell how common each residue is overall in the MSA. They’re normalized to sum to 1.
D 0.1439
E 0.3220
H 0.0495
K 0.3756
R 0.1090

Pairwise probabilities (normalized substitution frequencies):
       D      E      H      K      R
D 0.0207 0.0463 0.0071 0.0540 0.0157
E 0.0463 0.1037 0.0159 0.1209 0.0351
H 0.0071 0.0159 0.0025 0.0186 0.0054
K 0.0540 0.1209 0.0186 0.1410 0.0409
R 0.0157 0.0351 0.0054 0.0409 0.0119

Log Odds Substitution Scores:
Positive scores = substitutions more frequent than expected by chance (conserved or chemically similar)
Negative scores = substitutions less frequent (chemically dissimilar or rare

/opt/miniconda3/envs/md-311/lib/python3.11/site-packages/Bio/Align/substitution_matrices/__init__.py:214: RuntimeWarning: divide by zero encountered in log2
  raw_results = super().__array_ufunc__(ufunc, method, *args, **kwargs)


<!-- RESULTS SECTION -->
<div style="background-color:#f8f9fb; padding:25px; border-radius:12px; box-shadow:0px 4px 8px rgba(0,0,0,0.08);">

  <h2 style="color:#2c3e50; font-family:'Helvetica Neue',sans-serif; font-size:28px; margin-bottom:20px;">
    Results
  </h2>

  <!-- SECTION 1 -->
  <h3 style="color:#7f8c8d; font-family:'Georgia',serif; font-weight:normal; font-size:22px; margin-top:10px;">
    1. Taxonomic Coverage:
  </h3>

  <p style="color:#34495e; font-family:'Arial',sans-serif; font-size:15px; line-height:1.6; text-indent:25px;">
    The filtered PF10613 dataset contained 100 sequences spanning 72 unique species tokens, representing a broad distribution across vertebrates. The largest groups were fish (~55%, 55/100), followed by amphibians, reptiles, birds, and mammals at lower frequencies. The representation of both teleost and tetrapod lineages indicates that the L-glutamate/glycine-binding domain of GluN1 (PF10613) is broadly conserved across vertebrates.<br><br>
  </p>
  <!-- SECTION 2 -->
  <h3 style="color:#7f8c8d; font-family:'Georgia',serif; font-weight:normal; font-size:22px; margin-top:10px;">
    2. Sequence Similarity & Redundancy:
  </h3>

  <p style="color:#34495e; font-family:'Arial',sans-serif; font-size:15px; line-height:1.6; text-indent:25px;">
    Mean per-sequence coverage was 0.67, indicating that approximately two-thirds of each sequence aligns to the full human GluN1 domain region. The median pairwise identity was 93.9% (IQR 5.2%), showing very high overall sequence conservation. The alignment included 52 columns with ≥50% gaps and 98 well-covered columns (≤20% gaps), suggesting that a majority of residues are aligned across most taxa, but with localized insertions or deletions at loop or peripheral regions.
<br>
Henikoff weighting (a position-based method) downweights sequences that dominate highly conserved columns and upweights those that differ in rare or informative positions. Neff quantifies the effective diversity in your multiple sequence alignment (MSA). A low Neff score indicates redundancy, suggesting that the alignment is taxonomically narrow or dominated by one clade. Coevolution signals become unreliable because there aren’t enough independent substitutions to correlate. The redundancy analysis using Henikoff weighting yielded Neff = 31.9 and Neff/L = 0.21, far below the threshold (≥2.0) typically required for robust coevolutionary inference. When reweighted by sequence similarity cutoffs (θ = 0.8–0.9), Neff values approached ~1.0, confirming that the alignment is dominated by closely related vertebrate sequences with limited independent evolutionary variation. This means that most sequences are too similar to each other to provide statistically independent signals.<br><br>
  </p>
  <!-- SECTION 3 -->
  <h3 style="color:#7f8c8d; font-family:'Georgia',serif; font-weight:normal; font-size:22px; margin-top:10px;">
    3. Phylogenetic Interpretation:
  </h3>

  <p style="color:#34495e; font-family:'Arial',sans-serif; font-size:15px; line-height:1.6; text-indent:25px;">
    The neighbor-joining tree (BLOSUM62) correctly grouped human GluN1 with other mammalian NMDA receptor subunits, closely clustering with dolphin, bovine, and macaque, while amphibian, reptilian, and teleost GluN1 homologs formed distinct outgroups. This topology confirms the expected vertebrate divergence pattern for ionotropic glutamate receptor subunits. The high pairwise identity across diverse taxa implies that GluN1 maintains L-glutamate and glycine co-agonist binding. The limited Neff and narrow taxonomic range confirms that GluN1 is structurally and functionally constrained and suggests it’s performing an essential role where most mutations are purged by selection. This is evidence that GluN1 plays critical role in NMDA receptor signaling. <br><br>
  </p>
  <!-- SECTION 4 -->
  <h3 style="color:#7f8c8d; font-family:'Georgia',serif; font-weight:normal; font-size:22px; margin-top:10px;">
    4. Overall Conservation:
  </h3>

  <p style="color:#34495e; font-family:'Arial',sans-serif; font-size:15px; line-height:1.6; text-indent:25px;">
    Across the 153 aligned positions, the mean Shannon entropy was 0.363 bits, indicating that most sites show limited variability. A total of 110 residues (72%) were classified as highly conserved (<0.3 bits), while only 3 residues exceeded the hypervariable threshold (>2.0 bits). The remaining 39 positions were moderately conserved. This pattern shows that the majority of residues remain nearly invariant across vertebrates, reflecting strong evolutionary pressure to preserve receptor structure and ligand coordination. <br><br>
  </p>
  <!-- SECTION 5 -->
  <h3 style="color:#7f8c8d; font-family:'Georgia',serif; font-weight:normal; font-size:22px; margin-top:10px;">
    5. Conserved vs Variable Regions:
  </h3>

  <p style="color:#34495e; font-family:'Arial',sans-serif; font-size:15px; line-height:1.6; text-indent:25px;">
    Highly conserved residues clustered into five main blocks, 117–151 and 79–113, each spanning ~35 residues, exhibited mean entropy values of 0.02–0.04 bits. Shorter conserved motifs occurred at 60–70, 40–43, and 13–15, each containing near-zero entropy. These regions correspond to core secondary-structure segments forming the inner cleft of the LBD, where glutamate and glycine binding occurs. In contrast, only four single-residue sites (positions 26, 47, 52, and 73) were classified as hypervariable (entropy > 2.0 bits). These residues are likely surface-exposed loops or flexible hinge points not directly involved in ligand binding or domain packing. Their variability suggests tolerated substitutions that do not disrupt receptor function.<br><br>
  </p>
  <!-- SECTION 6 -->
  <h3 style="color:#7f8c8d; font-family:'Georgia',serif; font-weight:normal; font-size:22px; margin-top:10px;">
    5. Hydrophobic Character:
  </h3>

  <p style="color:#34495e; font-family:'Arial',sans-serif; font-size:15px; line-height:1.6; text-indent:25px;">
    MSA of residues 81–115 showed a consensus sequence perfectly conserved (100% identity) across 12 representative vertebrate sequences. The mean Kyte-Doolittle hydropathy score (−0.81) indicates a moderately hydrophilic region consistent with solvent-accessible loop or ligand-contact residues. MSA of residues 121–155 showed consensus also 100% conserved with mean KD ≈ −0.18, suggesting a polar-charged interface region possibly participating in subunit interactions or co-agonist stabilization. The consensus in these regions indicate a functional binding pocket and adjacent secondary-structure elements which are nearly invariant across vertebrates.<br><br>
  </p>
  <!-- SECTION 7 -->
  <h3 style="color:#7f8c8d; font-family:'Georgia',serif; font-weight:normal; font-size:22px; margin-top:10px;">
    6. Substitution Matrix:
  </h3>

  <p style="color:#34495e; font-family:'Arial',sans-serif; font-size:15px; line-height:1.6; text-indent:25px;">
    To quantify residue-level tolerance, a substitution matrix was derived from the observed replacement frequencies of polar/charged residues (D, E, H, K, R) across the alignment. The raw counts were dominated by K (37.6%) and E (32.2%), reflecting the prevalence of positively and negatively charged side chains in this domain. Strongly positive diagonal values (D: +2.8, H: +4.3, R: +3.1) indicate that these residues are highly self-conserved. Strongly negative off-diagonals (e.g., E↔K = −14.6, D↔E = −8.2) show that acidic and basic swaps are rare and likely disruptive. This pattern confirms the conservation of acidic residues substitute mostly within their own type, and basic residues do the same. This is consistent with preservation of electrostatic properties important for an ion-gated receptor.
  </p>

<!-- Analysis SECTION -->
<div style="background-color:#f8f9fb; padding:25px; border-radius:12px; box-shadow:0px 4px 8px rgba(0,0,0,0.08);">

  <h2 style="color:#2c3e50; font-family:'Helvetica Neue',sans-serif; font-size:28px; margin-bottom:20px;">
    Analysis
  </h2>

  <p style="color:#34495e; font-family:'Arial',sans-serif; font-size:15px; line-height:1.6; text-indent:25px;">
    Overall, the results demonstrate that the GluN1 ligand-binding domain is one of the most evolutionarily conserved regions among neurotransmitter-gated ion channels. Despite broad taxonomic representation, sequence diversity is minimal, indicating strong purifying selection to maintain the structural and electrostatic features required for receptor function. The alignment and entropy analyses show that conserved residues cluster in regions corresponding to ligand coordination and subunit-interface contacts, while variable positions are confined to peripheral, likely flexible loops. Substitution patterns confirm a bias toward chemically conservative changes, particularly among charged residues, highlighting the importance of preserving charge distribution and polarity for ligand binding and gating stability. Together, these findings support the view that GluN1 has remained under strict structural and functional constraints throughout vertebrate evolution, reflecting its essential role in NMDA receptor assembly and signaling.<br><br>
  </p>

<!-- Next Steps -->
<div style="background-color:#f8f9fb; padding:25px; border-radius:12px; box-shadow:0px 4px 8px rgba(0,0,0,0.08);">

  <h2 style="color:#2c3e50; font-family:'Helvetica Neue',sans-serif; font-size:28px; margin-bottom:20px;">
    Next Steps & Future Directions 
  </h2>

  <!-- Future Questions:
    1. Do conserved positions cluster near known functional sites (ligand-binding, gating, interfaces)?
           2. Are there motifs (e.g., ligand-binding signatures) that define subfamilies?
      3. Which residue pairs co-evolve? Do they correspond to contacts in 3D (7EU8)?
           4. Is there clear subfamily structure? Any lineage-specific specialization that should be treated separately?
-->

  <p style="color:#34495e; font-family:'Arial',sans-serif; font-size:15px; line-height:1.6; text-indent:25px;">
    <b>Assess clustering of conserved residues near functional sites</b><br>
Map low-entropy positions from the conservation table onto the 3D structure of GluN1 (PDB 7EU8) using PyMOL or ChimeraX. Overlay known ligand-binding and gating residues from literature annotations. Measure spatial proximity between conserved residues and known ligand-binding atoms to determine whether conservation correlates with catalytic or structural importance.
<br><br>
<b>Identify motifs defining subfamilies or receptor classes</b><br>
Extract consensus sequences from each major clade in the phylogenetic tree (e.g., mammals, amphibians, fish). Use MEME or WebLogo to visualize conserved motifs. Compare these patterns to known glutamate receptor signatures to detect subfamily-specific residue differences or potential adaptive variations.
<br><br>
<b>Detect co-evolving residue pairs and structural coupling</b><br>
Apply a coevolutionary analysis tool such as EVcouplings or GREMLIN using the cleaned MSA. Generate a contact map of top-scoring residue pairs and compare them to inter-residue contacts in 7EU8 using a distance threshold (<8 Å between Cα atoms). Evaluate whether co-evolving pairs align with intra- or inter-domain interfaces that may mediate allosteric communication.
<br><br>
<b>Characterize subfamily and lineage-specific specialization</b><br>
Perform sequence clustering (e.g., UPGMA or k-means on pairwise identity) to separate potential subgroups. Reconstruct sub-trees for each cluster and calculate entropy profiles independently. Examine whether certain taxa show distinct substitution patterns or motif variants, which could reflect functional adaptation or species-specific receptor tuning.<br><br>
  </p>

<!-- REFERENCES SECTION -->
<div style="background-color:#f8f9fb; padding:25px; border-radius:12px; box-shadow:0px 4px 8px rgba(0,0,0,0.08);">

  <h2 style="color:#2c3e50; font-family:'Helvetica Neue',sans-serif; font-size:28px; margin-bottom:20px;">
    References
  </h2>

  <!-- SECTION 1 -->
  <h3 style="color:#7f8c8d; font-family:'Georgia',serif; font-weight:normal; font-size:22px; margin-top:10px;">
    Tools:
  </h3>

  <p style="color:#34495e; font-family:'Arial',sans-serif; font-size:15px; line-height:1.6; text-indent:25px;">
    Biopython Tutorial & Cookbook — Biopython 1.85 documentation. (2025) https://biopython.org/docs/latest/Tutorial/index.html <br><br>
  </p>
  <h3 style="color:#7f8c8d; font-family:'Georgia',serif; font-weight:normal; font-size:22px; margin-top:10px;">
    Database Entries:
  </h3>

  <p style="color:#34495e; font-family:'Arial',sans-serif; font-size:15px; line-height:1.6; text-indent:25px;">
     Glutamate receptor ionotropic, NMDA 1 (Q05586) -   protein - InterPro. (2020, November 1). InterPro.https://www.ebi.ac.uk/interpro/protein/UniProt/Q05586/#table <br>
  </p>
  </p>